# LA 19 1s 

In [1]:
import pyaudio
import matplotlib.pylab as plt
import matplotlib
import torchaudio
import io
import numpy as np
import torch

torchaudio.set_audio_backend("soundfile")
torch.set_num_threads(5)

/tmp/ipykernel_1679916/179070195.py:9: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("soundfile")


## VAD

In [2]:
vad_model, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad',
                              model='silero_vad',
                              force_reload=True)

Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /home/hungdx/.cache/torch/hub/master.zip


In [3]:
(get_speech_timestamps,
 save_audio,
 read_audio,
 VADIterator,
 collect_chunks) = utils

## LA19_TRAIN_PROTOCOL

In [4]:
# import os
# import librosa
# import torchaudio
# from tqdm.notebook import tqdm

# LA19_DATASET_PATH = "/home/hungdx/Datasets/ASVspoof2019_LA_train"
# LA19_TRAIN_PROTOCOL_PATH = "/datad/hungdx/KDW2V-AASISTL/protocols/ASVspoof_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
# SAMPLE_RATE = 16000
# CUT_SIZE = 16000
# DURATION_CUT = 1
# DESTINATION_PATH = f"/datad/hungdx/Rawformer-implementation-anti-spoofing/datasets/ASVspoof2019_LA_train_{DURATION_CUT}s"
# DESTINATION_LA19_TRAIN_PROTOCOL_PATH = f"/datad/hungdx/Rawformer-implementation-anti-spoofing/datasets/protocols/ASVspoof2019.LA.cm.train.trn_{DURATION_CUT}s.txt"

# if os.path.exists(DESTINATION_LA19_TRAIN_PROTOCOL_PATH):
#     os.remove(DESTINATION_LA19_TRAIN_PROTOCOL_PATH)

# if not os.path.exists(DESTINATION_PATH):
#     os.makedirs(DESTINATION_PATH, exist_ok=True)

# THRESHOLD = 0.6  # Threshold for VAD to determine speech


# for index, line in tqdm(enumerate(open(LA19_TRAIN_PROTOCOL_PATH).readlines())):


#     line = line.replace('\n', '').split(' ')
#     file, attack_type, label = line[1], line[3], line[4] # Get file name, attack type, and label

#     file_path = os.path.join(LA19_DATASET_PATH, file + ".flac")
#     # waveform, sample_rate = librosa.load(file_path, sr=SAMPLE_RATE)
#     waveform, sample_rate = torchaudio.load(file_path)
#     waveform = waveform.squeeze(0)
#     # waveform, sample_rate = librosa.load(file_path, sr=SAMPLE_RATE)
#     # waveform = torch.from_numpy(waveform)
    

#     if sample_rate != SAMPLE_RATE:
#         raise ValueError(f"file {file}.flac sample rate of {file} is not {SAMPLE_RATE}")

#     duration = len(waveform) / SAMPLE_RATE

#     NUMBER_OF_CUTS = len(waveform) // CUT_SIZE # Number of cuts
#     #print(f"File: {file}, Duration: {duration}, Number of cuts: {NUMBER_OF_CUTS}")
#     RESIDUAL = len(waveform) % CUT_SIZE # Residual part

#     # Cut audio
#     for i in range(NUMBER_OF_CUTS):
#         start_from_in_sec = i * CUT_SIZE / SAMPLE_RATE
#         end_at_in_sec = (i + 1) * CUT_SIZE / SAMPLE_RATE

#         cut_waveform = waveform[i*CUT_SIZE:(i+1)*CUT_SIZE]
#         new_confidence = vad_model(cut_waveform, SAMPLE_RATE).item()
#         #print(f"File: {file}, part {i + 1}, Confidence: {new_confidence}, Start: {start_from_in_sec}, End: {end_at_in_sec}, Duration: {duration}")
            
#         # Get speech timestamps from start speech to end speech
#         # speech_timestamps = get_speech_timestamps(
#         #         cut_waveform, vad_model, sampling_rate=SAMPLE_RATE, min_speech_duration_ms=100, visualize_probs=True)
            
#         # print(speech_timestamps)

#         cut_waveform = cut_waveform.unsqueeze(0)
#         speech_file_name = f"{file}_{i}.flac"
#         no_speech_file_name = f"{file}_{i}_no_speech.flac"
#         speech_file_path = os.path.join(DESTINATION_PATH, speech_file_name)
#         no_speech_file_path = os.path.join(
#             DESTINATION_PATH, no_speech_file_name)

#         if new_confidence > THRESHOLD:
#             # Save audio
#             torchaudio.save(
#                 speech_file_path, cut_waveform, SAMPLE_RATE)

#             with open(DESTINATION_LA19_TRAIN_PROTOCOL_PATH, 'a+') as f:
#                 f.write(
#                     f"{line[0]} {speech_file_name.split('.flac')[0]} {line[2]} {attack_type} {label}\n")
                
#         else:
#             # Save audio
#             #print(f"Saving {no_speech_file_path}")
#             torchaudio.save(
#                 no_speech_file_path, cut_waveform, SAMPLE_RATE)
#             with open(DESTINATION_LA19_TRAIN_PROTOCOL_PATH, 'a+') as f:
#                 f.write(
#                     f"{line[0]} {no_speech_file_name.split('.flac')[0]} {line[2]} {attack_type} {label}\n")
            


#     ## Process residual part
#     if RESIDUAL > 0:
#         start_from_in_sec = NUMBER_OF_CUTS * CUT_SIZE / SAMPLE_RATE
#         end_at_in_sec = duration
#         cut_waveform = waveform[-RESIDUAL:]

#         # Check if cut_waveform is too short 
#         if len(cut_waveform) < 512: # 512 samples = 32ms
#             continue
        
#         new_confidence = vad_model(cut_waveform, SAMPLE_RATE).item()
#         #print(f"File: {file}, part {NUMBER_OF_CUTS + 1}, Confidence: {new_confidence}, Start: {start_from_in_sec}, End: {end_at_in_sec}, Duration: {duration}")

#         # Get speech timestamps from start speech to end speech
#         # speech_timestamps = get_speech_timestamps(
#         #         cut_waveform, vad_model, sampling_rate=SAMPLE_RATE, min_speech_duration_ms=100, visualize_probs=True)

#         # print(speech_timestamps)
#         cut_waveform = cut_waveform.unsqueeze(0)
#         speech_file_name = f"{file}_{NUMBER_OF_CUTS}_residual.flac"
#         no_speech_file_name = f"{file}_{NUMBER_OF_CUTS}_no_speech_residual.flac"

#         speech_file_path = os.path.join(DESTINATION_PATH, speech_file_name)
#         no_speech_file_path = os.path.join(
#             DESTINATION_PATH, no_speech_file_name)

#         if new_confidence > THRESHOLD:
#             # Save audio
#             torchaudio.save(
#                 speech_file_path, cut_waveform, SAMPLE_RATE)
#             with open(DESTINATION_LA19_TRAIN_PROTOCOL_PATH, 'a+') as f:
#                 f.write(
#                     f"{line[0]} {speech_file_name.split('.flac')[0]} {line[2]} {attack_type} {label}\n")
#         else:
#             # Save audio
#             torchaudio.save(
#                 no_speech_file_path, cut_waveform, SAMPLE_RATE)
#             with open(DESTINATION_LA19_TRAIN_PROTOCOL_PATH, 'a+') as f:
#                 f.write(
#                     f"{line[0]} {no_speech_file_name.split('.flac')[0]} {line[2]} {attack_type} {label}\n")
        

## LA19_DEV_PROTOCOL

In [5]:
import os
import librosa
import torchaudio
from tqdm.notebook import tqdm

LA19_DATASET_PATH = "/home/hungdx/Datasets/ASVspoof2019_LA_dev"
LA19_DEV_PROTOCOL_PATH = "/datad/hungdx/KDW2V-AASISTL/protocols/ASVspoof_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt"
SAMPLE_RATE = 16000
CUT_SIZE = 16000
DURATION_CUT = 1
DESTINATION_PATH = f"/datad/hungdx/Rawformer-implementation-anti-spoofing/datasets/ASVspoof2019_LA_dev_{DURATION_CUT}s"
DESTINATION_LA19_DEV_PROTOCOL_PATH = f"/datad/hungdx/Rawformer-implementation-anti-spoofing/datasets/protocols/ASVspoof2019.LA.cm.dev.trl{DURATION_CUT}s.txt"

if os.path.exists(DESTINATION_LA19_DEV_PROTOCOL_PATH):
    os.remove(DESTINATION_LA19_DEV_PROTOCOL_PATH)

if not os.path.exists(DESTINATION_PATH):
    os.makedirs(DESTINATION_PATH, exist_ok=True)

THRESHOLD = 0.6  # Threshold for VAD to determine speech


for index, line in tqdm(enumerate(open(LA19_DEV_PROTOCOL_PATH).readlines())):

    line = line.replace('\n', '').split(' ')
    # Get file name, attack type, and label
    file, attack_type, label = line[1], line[3], line[4]

    file_path = os.path.join(LA19_DATASET_PATH, file + ".flac")
    # waveform, sample_rate = librosa.load(file_path, sr=SAMPLE_RATE)
    waveform, sample_rate = torchaudio.load(file_path)
    waveform = waveform.squeeze(0)
    # waveform, sample_rate = librosa.load(file_path, sr=SAMPLE_RATE)
    # waveform = torch.from_numpy(waveform)

    if sample_rate != SAMPLE_RATE:
        raise ValueError(
            f"file {file}.flac sample rate of {file} is not {SAMPLE_RATE}")

    duration = len(waveform) / SAMPLE_RATE

    NUMBER_OF_CUTS = len(waveform) // CUT_SIZE  # Number of cuts
    # print(f"File: {file}, Duration: {duration}, Number of cuts: {NUMBER_OF_CUTS}")
    RESIDUAL = len(waveform) % CUT_SIZE  # Residual part

    # Cut audio
    for i in range(NUMBER_OF_CUTS):
        start_from_in_sec = i * CUT_SIZE / SAMPLE_RATE
        end_at_in_sec = (i + 1) * CUT_SIZE / SAMPLE_RATE

        cut_waveform = waveform[i*CUT_SIZE:(i+1)*CUT_SIZE]
        new_confidence = vad_model(cut_waveform, SAMPLE_RATE).item()
        # print(f"File: {file}, part {i + 1}, Confidence: {new_confidence}, Start: {start_from_in_sec}, End: {end_at_in_sec}, Duration: {duration}")

        # Get speech timestamps from start speech to end speech
        # speech_timestamps = get_speech_timestamps(
        #         cut_waveform, vad_model, sampling_rate=SAMPLE_RATE, min_speech_duration_ms=100, visualize_probs=True)

        # print(speech_timestamps)

        cut_waveform = cut_waveform.unsqueeze(0)
        speech_file_name = f"{file}_{i}.flac"
        no_speech_file_name = f"{file}_{i}_no_speech.flac"
        speech_file_path = os.path.join(DESTINATION_PATH, speech_file_name)
        no_speech_file_path = os.path.join(
            DESTINATION_PATH, no_speech_file_name)

        if new_confidence > THRESHOLD:
            # Save audio
            torchaudio.save(
                speech_file_path, cut_waveform, SAMPLE_RATE)

            with open(DESTINATION_LA19_DEV_PROTOCOL_PATH, 'a+') as f:
                f.write(
                    f"{line[0]} {speech_file_name.split('.flac')[0]} {line[2]} {attack_type} {label}\n")

        else:
            # Save audio
            # print(f"Saving {no_speech_file_path}")
            torchaudio.save(
                no_speech_file_path, cut_waveform, SAMPLE_RATE)
            with open(DESTINATION_LA19_DEV_PROTOCOL_PATH, 'a+') as f:
                f.write(
                    f"{line[0]} {no_speech_file_name.split('.flac')[0]} {line[2]} {attack_type} {label}\n")

    # Process residual part
    if RESIDUAL > 0:
        start_from_in_sec = NUMBER_OF_CUTS * CUT_SIZE / SAMPLE_RATE
        end_at_in_sec = duration
        cut_waveform = waveform[-RESIDUAL:]

        # Check if cut_waveform is too short
        if len(cut_waveform) < 512:  # 512 samples = 32ms
            continue

        new_confidence = vad_model(cut_waveform, SAMPLE_RATE).item()
        # print(f"File: {file}, part {NUMBER_OF_CUTS + 1}, Confidence: {new_confidence}, Start: {start_from_in_sec}, End: {end_at_in_sec}, Duration: {duration}")

        # Get speech timestamps from start speech to end speech
        # speech_timestamps = get_speech_timestamps(
        #         cut_waveform, vad_model, sampling_rate=SAMPLE_RATE, min_speech_duration_ms=100, visualize_probs=True)

        # print(speech_timestamps)
        cut_waveform = cut_waveform.unsqueeze(0)
        speech_file_name = f"{file}_{NUMBER_OF_CUTS}_residual.flac"
        no_speech_file_name = f"{file}_{NUMBER_OF_CUTS}_no_speech_residual.flac"

        speech_file_path = os.path.join(DESTINATION_PATH, speech_file_name)
        no_speech_file_path = os.path.join(
            DESTINATION_PATH, no_speech_file_name)

        if new_confidence > THRESHOLD:
            # Save audio
            torchaudio.save(
                speech_file_path, cut_waveform, SAMPLE_RATE)
            with open(DESTINATION_LA19_DEV_PROTOCOL_PATH, 'a+') as f:
                f.write(
                    f"{line[0]} {speech_file_name.split('.flac')[0]} {line[2]} {attack_type} {label}\n")
        else:
            # Save audio
            torchaudio.save(
                no_speech_file_path, cut_waveform, SAMPLE_RATE)
            with open(DESTINATION_LA19_DEV_PROTOCOL_PATH, 'a+') as f:
                f.write(
                    f"{line[0]} {no_speech_file_name.split('.flac')[0]} {line[2]} {attack_type} {label}\n")

0it [00:00, ?it/s]

## LA19_TRAIN_PROTOCOL PARALLEL CODE

In [6]:
import os
# import torchaudio
# from tqdm.notebook import tqdm
# from concurrent.futures import ThreadPoolExecutor

# LA19_DATASET_PATH = "/home/hungdx/Datasets/ASVspoof2019_LA_train"
# LA19_TRAIN_PROTOCOL_PATH = "/datad/hungdx/KDW2V-AASISTL/protocols/ASVspoof_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
# SAMPLE_RATE = 16000
# CUT_SIZE = 16000
# DURATION_CUT = 1
# DESTINATION_PATH = f"/datad/hungdx/Rawformer-implementation-anti-spoofing/datasets/ASVspoof2019_LA_train_{DURATION_CUT}s"
# DESTINATION_LA19_TRAIN_PROTOCOL_PATH = f"/datad/hungdx/Rawformer-implementation-anti-spoofing/datasets/protocols/ASVspoof2019.LA.cm.train.trn_{DURATION_CUT}s.txt"

# if os.path.exists(DESTINATION_LA19_TRAIN_PROTOCOL_PATH):
#     os.remove(DESTINATION_LA19_TRAIN_PROTOCOL_PATH)
# if not os.path.exists(DESTINATION_PATH):
#     os.makedirs(DESTINATION_PATH, exist_ok=True)

# THRESHOLD = 0.6  # Threshold for VAD to determine speech


# def process_audio_file(line):
#     line = line.strip().split(' ')
#     file, attack_type, label = line[1], line[3], line[4]

#     file_path = os.path.join(LA19_DATASET_PATH, file + ".flac")
#     waveform, sample_rate = torchaudio.load(file_path)
#     waveform = waveform.squeeze(0)

#     if sample_rate != SAMPLE_RATE:
#         raise ValueError(
#             f"file {file}.flac sample rate of {file} is not {SAMPLE_RATE}")

#     NUMBER_OF_CUTS = len(waveform) // CUT_SIZE
#     RESIDUAL = len(waveform) % CUT_SIZE
#     outputs = []

#     for i in range(NUMBER_OF_CUTS):
#         cut_waveform = waveform[i*CUT_SIZE:(i+1)*CUT_SIZE]
#         new_confidence = vad_model(cut_waveform, SAMPLE_RATE).item()
#         file_suffix = "no_speech" if new_confidence <= THRESHOLD else ""
#         file_name = f"{file}_{i}_{file_suffix}.flac"
#         torchaudio.save(os.path.join(DESTINATION_PATH, file_name),
#                         cut_waveform.unsqueeze(0), SAMPLE_RATE)
#         outputs.append(
#             f"{line[0]} {file_name.split('.flac')[0]} {line[2]} {attack_type} {label}\n")

#     if RESIDUAL > 0:
#         if len(waveform[-RESIDUAL:]) >= 512:
#             process_residual_part(waveform, RESIDUAL,
#                                   NUMBER_OF_CUTS, file, line, outputs)

#     return outputs


# def process_residual_part(waveform, RESIDUAL, NUMBER_OF_CUTS, file, line, outputs):
#     cut_waveform = waveform[-RESIDUAL:]
#     new_confidence = vad_model(cut_waveform, SAMPLE_RATE).item()
#     file_suffix = "no_speech" if new_confidence <= THRESHOLD else ""
#     file_name = f"{file}_{NUMBER_OF_CUTS}_{file_suffix}.flac"
#     torchaudio.save(os.path.join(DESTINATION_PATH, file_name),
#                     cut_waveform.unsqueeze(0), SAMPLE_RATE)
#     outputs.append(
#         f"{line[0]} {file_name.split('.flac')[0]} {line[2]} {attack_type} {label}\n")


# lines = open(LA19_TRAIN_PROTOCOL_PATH).readlines()
# with ThreadPoolExecutor(max_workers=10) as executor:
#     futures = [executor.submit(process_audio_file, line) for line in lines]
#     results = [future.result() for future in tqdm(futures, total=len(futures))]

# with open(DESTINATION_LA19_TRAIN_PROTOCOL_PATH, 'a') as f:
#     for result in results:
#         for line in result:
#             f.write(line)# 